<a href="https://colab.research.google.com/github/Anakad1/bootcampgenai/blob/main/exxpw2d3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder



In [5]:
df = sns.load_dataset("titanic")

In [6]:
print(df.head())
print(df.info())

   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-nu

exercice 1

In [7]:
# Vérifier s'il y a des doublons
duplicates = df.duplicated().sum()
print(f"\n Nombre de doublons détectés : {duplicates}")

# Supprimer les doublons si présents
df = df.drop_duplicates()

# Vérifier la suppression
print(f"\n Nombre de lignes après suppression des doublons : {df.shape[0]}")



 Nombre de doublons détectés : 107

 Nombre de lignes après suppression des doublons : 784


exercice 2

In [8]:
# Identifier les valeurs manquantes
missing_values = df.isnull().sum()
print("\n Valeurs manquantes par colonne :\n", missing_values)

# Remplissage des valeurs manquantes selon la nature des données
df['age'].fillna(df['age'].median(), inplace=True)  # Remplissage par la médiane
df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)  # Remplissage par la valeur la plus fréquente
df.drop(columns=['deck'], inplace=True)  # Suppression d'une colonne avec trop de valeurs manquantes

# Vérifier après traitement
print("\n Valeurs manquantes après traitement :\n", df.isnull().sum())



 Valeurs manquantes par colonne :
 survived         0
pclass           0
sex              0
age            106
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           582
embark_town      2
alive            0
alone            0
dtype: int64

 Valeurs manquantes après traitement :
 survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    2
alive          0
alone          0
dtype: int64


<ipython-input-8-b9333f0cb152>:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['age'].fillna(df['age'].median(), inplace=True)  # Remplissage par la médiane
<ipython-input-8-b9333f0cb152>:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(v

exercice 3

In [11]:
# Création de la nouvelle feature "Family Size" (nombre total de membres de la famille à bord)
df['family_size'] = df['sibsp'] + df['parch'] + 1

# Before extracting the title, check if the 'name' column exists
if 'name' in df.columns:
    # Extraction du titre depuis la colonne "name"
    df['title'] = df['name'].apply(lambda x: x.split(',')[1].split('.')[0].strip())
else:
    # Handle the case where 'name' column is missing
    # For example, you could skip this step or create a default title
    print("Warning: 'name' column not found. Skipping title extraction.")
    df['title'] = 'Unknown'  # or any other suitable default

# Encodage one-hot des variables catégorielles
df = pd.get_dummies(df, columns=['sex', 'embarked', 'title'], drop_first=True)

# Afficher les nouvelles colonnes
print("\n Aperçu des nouvelles fonctionnalités :\n", df.head())



 Aperçu des nouvelles fonctionnalités :
    survived  pclass   age  sibsp  parch     fare  class    who  adult_male  \
0         0       3  22.0      1      0   7.2500  Third    man        True   
1         1       1  38.0      1      0  71.2833  First  woman       False   
2         1       3  26.0      0      0   7.9250  Third  woman       False   
3         1       1  35.0      1      0  53.1000  First  woman       False   
4         0       3  35.0      0      0   8.0500  Third    man        True   

   embark_town alive  alone  family_size  sex_male  embarked_Q  embarked_S  
0  Southampton    no  False            2      True       False        True  
1    Cherbourg   yes  False            2     False       False       False  
2  Southampton   yes   True            1     False       False        True  
3  Southampton   yes  False            2     False       False        True  
4  Southampton    no   True            1      True       False        True  


exercice 4

In [12]:
# Détection des valeurs aberrantes avec IQR pour la colonne "fare"
Q1 = df['fare'].quantile(0.25)
Q3 = df['fare'].quantile(0.75)
IQR = Q3 - Q1

# Définition des bornes pour détecter les outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Supprimer les valeurs aberrantes de "fare"
df = df[(df['fare'] >= lower_bound) & (df['fare'] <= upper_bound)]

# Vérifier après suppression
print(f"\n Nombre de lignes après suppression des valeurs aberrantes : {df.shape[0]}")



 Nombre de lignes après suppression des valeurs aberrantes : 682


exercixce 5

In [13]:
scaler = MinMaxScaler()

# Normalisation des colonnes numériques
df[['age', 'fare']] = scaler.fit_transform(df[['age', 'fare']])

# Vérification des nouvelles échelles de valeurs
print("\n Aperçu des données normalisées :\n", df[['age', 'fare']].head())



 Aperçu des données normalisées :
         age      fare
0  0.271174  0.101707
1  0.472229  1.000000
2  0.321438  0.111176
3  0.434531  0.744915
4  0.434531  0.112930


exercice 6

In [14]:
# Encodage des variables catégorielles (one-hot et label encoding)
label_encoder = LabelEncoder()
df['class'] = label_encoder.fit_transform(df['class'])  # Encodage d'étiquette

# Vérifier les colonnes encodées
print("\n Aperçu des colonnes encodées :\n", df.head())



 Aperçu des colonnes encodées :
    survived  pclass       age  sibsp  parch      fare  class    who  \
0         0       3  0.271174      1      0  0.101707      2    man   
1         1       1  0.472229      1      0  1.000000      0  woman   
2         1       3  0.321438      0      0  0.111176      2  woman   
3         1       1  0.434531      1      0  0.744915      0  woman   
4         0       3  0.434531      0      0  0.112930      2    man   

   adult_male  embark_town alive  alone  family_size  sex_male  embarked_Q  \
0        True  Southampton    no  False            2      True       False   
1       False    Cherbourg   yes  False            2     False       False   
2       False  Southampton   yes   True            1     False       False   
3       False  Southampton   yes  False            2     False       False   
4        True  Southampton    no   True            1      True       False   

   embarked_S  
0        True  
1       False  
2        True  
3     

exercice 7

In [15]:
# Création de groupes d'âge (enfants, jeunes adultes, adultes, seniors)
df['age_group'] = pd.cut(df['age'], bins=[0, 0.2, 0.5, 0.8, 1], labels=['child', 'young_adult', 'adult', 'senior'])

# Encodage one-hot des groupes d'âge
df = pd.get_dummies(df, columns=['age_group'], drop_first=True)

# Vérification des nouvelles colonnes
print("\n✅ Aperçu des nouvelles colonnes d'âge :\n", df.head())



✅ Aperçu des nouvelles colonnes d'âge :
    survived  pclass       age  sibsp  parch      fare  class    who  \
0         0       3  0.271174      1      0  0.101707      2    man   
1         1       1  0.472229      1      0  1.000000      0  woman   
2         1       3  0.321438      0      0  0.111176      2  woman   
3         1       1  0.434531      1      0  0.744915      0  woman   
4         0       3  0.434531      0      0  0.112930      2    man   

   adult_male  embark_town alive  alone  family_size  sex_male  embarked_Q  \
0        True  Southampton    no  False            2      True       False   
1       False    Cherbourg   yes  False            2     False       False   
2       False  Southampton   yes   True            1     False       False   
3       False  Southampton   yes  False            2     False       False   
4        True  Southampton    no   True            1      True       False   

   embarked_S  age_group_young_adult  age_group_adult  age_gro